# Lab 20 · When the intent is wrong

**Time:** 45 min · **Work in teams of 3**

**Where we are in the day:** Lab 19 showed that the small model is fast for many people at once, with about 89 per cent exact answers; so about 1 message in 10 is still wrong.
**Today's question:** can the model tell us when it is not sure, so that the assistant asks back instead of guessing?
**What you will do:**
- Read the evidence for one mistake, and measure how sure the model is about each of the 72 messages.
- Choose a threshold: answer only when the model is sure enough, otherwise ask the user back.
- Try a fallback to a bigger model for the unsure messages, and save the mistakes for the next training.

**What you will see:** in our test run the model was more sure when it was right (average 0.973) than when it was wrong (0.704), and a threshold of 0.9 made every answer we gave right.
**Why it matters on a device:** a device cannot call a big model every time, so it needs a cheap way to know when to ask the user or escalate.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **30 seconds** in all, once the models are on the disk.
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer. The first run downloads the 3-billion-parameter model of Day 2 (several GB) if it is not there yet; this can take a few minutes, and the trainer can tell you if it is normal.
- Before you start: Labs 17 and 18 must have been run, because the first cell loads the merged model that Lab 18 creates. The scoreboard should have rows 1 to 4 (Lab 19 saved the fourth one).

**Words used in this lab:**
- **Probability:** a number from 0 to 1 for how likely the model thinks an answer is. 1 means certain.
- **Confidence:** the probability the model gives to its own answer. It is the model's own sureness, not a proof.
- **Threshold:** the line we draw. At or above it the assistant answers; below it, it asks back.
- **Fallback:** the second choice when the first one is not enough, here a bigger model.

**No model is always right.** A voice assistant that guesses wrongly turns on the wrong thing. One that says "did you mean this or that?" at the right moment is much better. The skill is knowing **when to trust the model** and **what to do when you cannot**.

**Python in this lab:** loops, `if`, lists and dictionaries. The probability code is in `edge_confidence.py`.

In [ ]:
import sys, os, json
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
import edge_confidence as ec
import home_manager as hm
from slm_lab import check, OUT
MERGED_FOLDER = str(OUT / "edge_merged")
golden = nlu.load_golden()

Now load the merged model and test it on the 72 messages.

In [ ]:
tok, model = el.load_edge_model(MERGED_FOLDER)
result = el.evaluate(tok, model, nlu.SHORT_SYSTEM)
print("fine-tuned model:", result["exact %"], "% exact on the 72 messages,", len(result["wrong"]), "wrong")

In [ ]:
check(len(golden) == 72 and os.path.exists(MERGED_FOLDER), "Checkpoint: the merged model is loaded and tested on the 72 messages.",
      "The merged model is missing. Run Labs 17 and 18 first, then restart the kernel here.")

## 20.1 Where are the mistakes?
Write down how many messages are wrong in total. In our test run the exact accuracy was about 89 per cent, which is about 8 wrong messages of 72 (it changes a little from run to run).

In [ ]:
wrong_by_kind = {}
for mistake in result["wrong"]:
    wrong_by_kind[mistake["kind"]] = wrong_by_kind.get(mistake["kind"], 0) + 1
print("wrong messages by kind:", wrong_by_kind)
for mistake in result["wrong"][:8]:
    print(nlu.describe_mistake(mistake))

**You should see** the list of what is still wrong (the first eight). Sort them into three groups in your head: a **wrong intent** (the assistant would do the wrong thing), a **wrong detail** (right intent, wrong room or value), and **messages outside its job** that it tried to answer anyway (for example a TV command in a home assistant).

## 20.2 Read the evidence first
When an answer is wrong, **do not blame the model first.** Read three things: **what text arrived**, **what the model was given**, and **what it said**. Many mistakes are a text mistake or a prompt mistake.

In [ ]:
def explain_one(text):
    prompt_text = el.make_prompt_text(tok, nlu.SHORT_SYSTEM, text)
    reply = el.ask_many(tok, model, [prompt_text])["replies"][0]
    print("1. text that arrived :", repr(text))
    print("2. what the model saw:", repr(prompt_text))
    print("3. what it said      :", reply)
    print("   code read intent  :", nlu.read_intent(reply), nlu.read_slots(reply))

explain_one(result["wrong"][0]["text"])

**You should see** the exact text, the exact prompt (with the chat-template markers) and the raw reply. Ask three questions: Was the text already damaged by speech recognition? Did the prompt contain what the model needed? Did the model ignore something it was given? **The answer decides the fix:** clean the text, change the prompt, or train with more examples.

## 20.3 How sure is the model?
A model can give you **more than an answer**: it gives the **probability** it assigns to each possible answer. A model that is sure puts almost all of the probability on one intent. A model that is guessing spreads it out. We call the probability of its chosen answer its **confidence**. *In one sentence: confidence is how sure the model says it is.*

For every message we ask: how likely is each of the nine intents? Then we look at the best one and the runner-up.

In [ ]:
rows = []
for case in golden:
    probabilities = ec.label_probabilities(tok, model, nlu.SHORT_SYSTEM, case["text"])
    best, best_p, second, second_p = ec.best_two(probabilities)
    rows.append({"text": case["text"], "kind": case["kind"], "expected": case["intent"], "best": best, "best_p": best_p,
                 "second": second, "second_p": second_p, "intent_right": best == case["intent"]})
for row in rows[:6]:
    print(row["text"], "->", row["best"], round(row["best_p"], 2), "| runner-up:", row["second"], round(row["second_p"], 2))

**You should see** probabilities close to 1 for easy messages and lower for hard ones.

**Predict.** Write on paper: on average, is the model **more** sure when it is right or when it is wrong? Write a number between 0 and 1 for each.

In [ ]:
sum_right = 0
count_right = 0
sum_wrong = 0
count_wrong = 0
for row in rows:
    if row["intent_right"]:
        sum_right = sum_right + row["best_p"]
        count_right = count_right + 1
    else:
        sum_wrong = sum_wrong + row["best_p"]
        count_wrong = count_wrong + 1
print("right:", count_right, "messages, average confidence", round(sum_right / count_right, 3))
print("wrong:", count_wrong, "messages, average confidence", round(sum_wrong / max(count_wrong, 1), 3))

**You should see** that the model is **less sure when it is wrong**. In our test run: an average confidence of 0.973 when right and 0.704 when wrong. That gap is what we use. It is not perfect: some wrong answers are given with high confidence, and these are the dangerous ones.

In [ ]:
check(len(rows) == 72, "Checkpoint: you have a confidence for each of the 72 messages.",
      "Run the cell that loops over the golden set again; it takes a few seconds.")

## 20.4 A rule: answer only when sure
The rule: if the best probability is **at least a threshold**, answer. Otherwise **ask the user back**. A high threshold asks back more often (safe, but annoying). A low threshold answers more often (smooth, but riskier). The table shows the trade for each threshold.

*In one sentence: the threshold decides how sure the model must be before it acts.*

In [ ]:
thresholds = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
table_rows = []

For each threshold we count the messages answered, how many of those are right, and how many good answers we held back.

In [ ]:
for threshold in thresholds:
    answered = 0
    answered_right = 0
    held_back_right = 0
    for row in rows:
        if row["best_p"] >= threshold:
            answered = answered + 1
            if row["intent_right"]:
                answered_right = answered_right + 1
        elif row["intent_right"]:
            held_back_right = held_back_right + 1
    table_rows.append({"threshold": threshold, "answered %": round(100 * answered / len(rows)),
                       "right among answered %": round(100 * answered_right / max(answered, 1), 1),
                       "good answers held back": held_back_right})
pd.DataFrame(table_rows)

**You should see** that as the threshold rises, the **accuracy of the answers we give goes up**, and the share of messages we answer **goes down**. In our test run: at 0.5 we answered 99 per cent of the messages and 95.8 per cent of those were right; at 0.9 we answered 86 per cent and all of them were right, holding back 6 good answers; at 0.99 we answered 82 per cent. There is no free lunch: every wrong answer we avoid costs some good answers that we hold back.

In [ ]:
check(len(table_rows) == 6, "Checkpoint: the threshold table has 6 rows.",
      "Run the table cell again. The cell with the thresholds list must be run first.")

## 20.5 Your turn: choose the threshold
Your product needs **at least 99 per cent right among the answered messages**. Which is the **lowest** threshold from the table that gives it? (The lowest threshold answers the most messages.) If none does, say `none`. Write your answer on paper first, then use the code. In our test run the answer was 0.9. If you are stuck, open the solution cell below the starter.

In [ ]:
chosen_threshold = 0.99                          # TODO: find the lowest threshold with "right among answered %" of at least 99
for row in table_rows:
    pass                                         # TODO: look at each row of the table
print("chosen threshold:", chosen_threshold)

In [ ]:
check(chosen_threshold == "none" or chosen_threshold in thresholds, "The threshold is one of the values of the table (or none).",
      "Pick one of the thresholds in the table, or the word none.")

## 20.6 Ask back, politely
When the model is not sure, the assistant should say **what it understood and offer the two best choices**. Here is how that sounds for the messages we hold back.

In [ ]:
NAMES = {"set_light": "switch a light", "set_ac": "set the air conditioner", "set_blinds": "move the blinds", "run_scene": "run a scene",
         "ask_manual": "answer a question about an appliance", "yes": "say yes", "no": "say no", "cancel": "cancel", "other": "do something else"}
THRESHOLD = 0.9                                 # used when the table had no threshold that reaches 99 %
if chosen_threshold != "none":
    THRESHOLD = chosen_threshold
shown = 0
for row in rows:
    if row["best_p"] < THRESHOLD and shown < 6:
        print('You said: "' + row["text"] + '"')
        print("   Assistant: Sorry, do you want me to " + NAMES[row["best"]] + ", or to " + NAMES[row["second"]] + "?")
        shown = shown + 1

**You should see** short, honest questions instead of a wrong action. For a risky action (open the door) you would ask even when the model is fairly sure: that is the human approval of Day 2.

In [ ]:
check(shown > 0, "Checkpoint: the assistant asked back for at least one message.",
      "No message is below the threshold. That is possible with a low threshold; try THRESHOLD = 0.95 and run the cell again.")

## 20.7 A fallback chain: a bigger model for the hard ones
Another answer to "not sure": ask a **bigger model**, but only for those messages. This second choice is the **fallback**. Most messages stay on the small, fast model. The big one is the **exception**, not the rule. *In one sentence: the small model answers first, and the big model only helps with the hard ones.*

This is how a system moves from the cloud to the edge without losing quality: the cloud model becomes the fallback.

We use the 3-billion-parameter model of Day 2 with Day 2's prompt, only on the held-back messages.

In [ ]:
held_back = []
for index in range(len(rows)):
    if rows[index]["best_p"] < THRESHOLD:
        held_back.append(index)
print("held back:", len(held_back), "of", len(rows), "messages")
del model
torch.cuda.empty_cache()
big_tok, big_model = el.load_edge_model("Qwen/Qwen2.5-3B-Instruct")
big_prompts = []
for index in held_back:
    big_prompts.append(el.make_prompt_text(big_tok, hm.UNDERSTAND_SYSTEM, golden[index]["text"]))
big_replies = el.ask_many(big_tok, big_model, big_prompts)["replies"]

In [ ]:
big_right = 0
for position in range(len(held_back)):
    if nlu.read_intent(big_replies[position]) == golden[held_back[position]]["intent"]:
        big_right = big_right + 1
small_right_on_held = 0
for index in held_back:
    if rows[index]["intent_right"]:
        small_right_on_held = small_right_on_held + 1
print("on the held-back messages: small model right", small_right_on_held, ", bigger model right", big_right, ", of", len(held_back))

**You should see** whether the bigger model helps on exactly the messages where the small one was unsure. In our test run, with a threshold of 0.9, 10 of the 72 messages were held back: the small model was right on 6 of them and the bigger model on 7. So the gain was small. **Be honest with the numbers:** if the bigger model is not clearly better there, the better answer is to ask the user, or to train the small model on these cases. A fallback is only worth its cost if it **fixes** something.

In [ ]:
final_right = 0
for index in range(len(rows)):
    if index in held_back:
        position = held_back.index(index)
        if nlu.read_intent(big_replies[position]) == golden[index]["intent"]:
            final_right = final_right + 1
    elif rows[index]["intent_right"]:
        final_right = final_right + 1
small_only_right = 0
for row in rows:
    if row["intent_right"]:
        small_only_right = small_only_right + 1
print("small model alone     : intent right", round(100 * small_only_right / len(rows), 1), "%")
print("small + bigger model  : intent right", round(100 * final_right / len(rows), 1), "%, bigger model used for", round(100 * len(held_back) / len(rows)), "% of messages")

**You should see** two numbers and a cost: the chain's accuracy, and the share of messages that needed the expensive model. In our test run: small model alone 94.4 per cent intent right, small plus bigger model 95.8 per cent, with the bigger model used for 14 per cent of the messages. (Here we count only the intent, not the slots.)

In [ ]:
check(small_only_right > 0 and final_right > 0, "Checkpoint: both the small model alone and the chain were scored.",
      "Run the two cells of 20.7 again, in order. If the bigger model did not load, ask the trainer.")

## 20.8 Close the loop: keep the mistakes
Every message the assistant was unsure about, or got wrong, is **a free training example for the next version**, once a person has put the right label on it. Save them.

In [ ]:
misses = []
for index in range(len(rows)):
    if rows[index]["best_p"] < THRESHOLD or not rows[index]["intent_right"]:
        misses.append({"text": rows[index]["text"], "model_said": rows[index]["best"], "confidence": round(rows[index]["best_p"], 3),
                       "correct_label": golden[index]["intent"]})
MISSES_FILE = str(OUT / "edge_misses.json")
with open(MISSES_FILE, "w", encoding="utf-8") as f:
    json.dump(misses, f, indent=1, ensure_ascii=False)
print("saved", len(misses), "messages for the next training:", MISSES_FILE)

In [ ]:
check(os.path.exists(MISSES_FILE), "Checkpoint: the mistakes are saved in a file.",
      "The file is missing. Run the cell above again.")

**You should see** a file of messages with the label a person would confirm. This is the loop of every real product: **measure, find what fails, add it to the training data, train again, measure again.** The golden set tells you whether the new version is really better.

In [ ]:
answered_right = 0
answered = 0
for row in rows:
    if row["best_p"] >= THRESHOLD:
        answered = answered + 1
        if row["intent_right"]:
            answered_right = answered_right + 1
el.save_row("5 safe", {"what": "ask back below confidence " + str(THRESHOLD), "answered_percent": round(100 * answered / len(rows)),
                      "right_among_answered_percent": round(100 * answered_right / max(answered, 1), 1)})
print(el.load_scoreboard()["5 safe"])
check("5 safe" in el.load_scoreboard(), "Checkpoint: the fifth row of the scoreboard is saved. Lab 21 can read all five rows.",
      "The row is missing. Run the cell above again.")

## 20.9 Your turn: your three hardest messages
Now try to **fool the model**. Write three messages that you think are hard: an ambiguous command, a TV command (this assistant does not control a TV), a mix of languages, or a message with a speech-recognition mistake. See how sure the model is, and whether your threshold would answer or ask back. This is how you choose a threshold for a real product: not from a graph, but from the messages that worry you.

In [ ]:
del big_model
torch.cuda.empty_cache()
tok, model = el.load_edge_model(MERGED_FOLDER)
print("the small model is loaded again")

In [ ]:
HARD_MESSAGES = [
    "change the channel to the news",       # an example: a TV command, not a job of this assistant
    "",                                      # TODO 1: a message you think will fool it
    "",                                      # TODO 2
    "",                                      # TODO 3
]
ec.show_confidence(tok, model, nlu.SHORT_SYSTEM, HARD_MESSAGES, THRESHOLD)

**You should see** for each message the best intent, how sure the model is, the runner-up, and the decision (`ANSWER` or `ASK BACK`). Look for the dangerous ones: messages where the model **answers** with high confidence but the answer is wrong. **Write one sentence: would you move the threshold for your product, and why?**

## What you take away
- Read the evidence before blaming the model: the text, the prompt and the reply.
- The model's confidence is lower when it is wrong. Answering only above a threshold trades smoothness for safety; in our test run a threshold of 0.9 made all the answers we gave right, and we asked back for 14 per cent.
- A bigger model as a fallback is worth it only if it fixes the hard cases (here it helped a little), and the mistakes you save are the next training set.

## Check your understanding
1. Why is "ask the user back" often better than guessing?
*✍️ Your answer:* 
2. You raise the threshold from 0.8 to 0.95. What happens to the accuracy of the answers and to the number of questions asked?
*✍️ Your answer:* 
3. When is a bigger-model fallback not worth it?
*✍️ Your answer:* 
4. What do you do with the messages the assistant got wrong?
*✍️ Your answer:* 

## Next
**Lab 21:** put every number of the day against the brief, and hear the assistant work on spoken commands.